<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c13_ex10_revisited.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 - Exercise 10 - revisited

Another approach to solve exercise 10 of chapter 13, with ideas for the [official solution](https://colab.research.google.com/github/ageron/handson-mlp/blob/main/13_processing_sequences_using_rnns_and_cnns.ipynb#scrollTo=vZtsoIf1IRkz$0).

**Main differences from previous approach**:
- Sequences of single notes, not of chorales
- Single notes are one-hot encoded
- sequence-to-sequence, not sequence-to-vector

**Assignment from the book**:

The Bach chorales dataset.
Composed of 382 chorales. Each chorale is 100 to 640 time steps long, and each time step contains 4 integers, where each integer corresponds to a note's index on a piano (except for the value 0, which means that no note is played).

Train a model - recurrent, convolutional or both - that can predict the next time step (four notes), given a sequence of time steps from a chorale.

Then use this model to generate Bach-like music, one note at a time:
you can do this by giving the model the start of a chorale and asking it to predict the next time step, then appending these time steps to the input sequence and asking the model for the next note, and so on.

Google's Coconet model was used for a nice Google doodle about Bach.

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

## Import the data

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales"
)

print(DATA_DIR.exists())

True


In [4]:
import pandas as pd

path = Path("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/chorale_000.csv")
df = pd.read_csv(path)
df.head()

,note0,note1,note2,note3
0,74,70,65,58
1,74,70,65,58
2,74,70,65,58
3,74,70,65,58
4,75,70,58,55


For a sequence-to-sequence approach, the target must contain tha data shifted by 1.

In [72]:
class BachDataset(torch.utils.data.Dataset):
  def __init__(self, series, window_length):
    self.series = series
    self.window_length = window_length

  def __len__(self):
    return len(self.series) - self.window_length

  def __getitem__(self,idx):
    if idx >= len(self):
      raise IndexError('dataset index out of range')
    end = idx + self.window_length
    window = self.series[idx : end]
    target = self.series[idx + 1: end + 1].argmax(dim=-1)
    return window, target

## Prepare the data
- import from the csv
- turn chorales into single notes
- one-hot encoding

In [6]:
import os

train_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train")
valid_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/valid")
test_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/test")

train_paths = []
valid_paths = []
test_paths = []

for name in train_list:
  train_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/"+name)
for name in valid_list:
  valid_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/valid/"+name)
for name in test_list:
  test_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/test/"+name)

Create a list of all chorales in the folder train

In [15]:
import csv

train_chorales_list = []
for path in train_paths:
  with open(path) as f:
    reader = csv.reader(f)
    next(reader)  # skip header row
    chorale = [[int(note) for note in row] for row in reader]
    train_chorales_list.append(torch.tensor(chorale))
# print chorales_000
print(train_chorales_list[0])

tensor([[67, 62, 55, 47],
        [67, 62, 55, 47],
        [67, 62, 55, 47],
        ...,
        [67, 59, 55, 43],
        [67, 59, 55, 43],
        [67, 59, 55, 43]])


List of chorales in valid and test folders

In [21]:
valid_chorales_list = []
for path in valid_paths:
  with open(path) as f:
    reader = csv.reader(f)
    next(reader)  # skip header row
    chorale = [[int(note) for note in row] for row in reader]
    valid_chorales_list.append(torch.tensor(chorale))

In [22]:
test_chorales_list = []
for path in test_paths:
  with open(path) as f:
    reader = csv.reader(f)
    next(reader)  # skip header row
    chorale = [[int(note) for note in row] for row in reader]
    test_chorales_list.append(torch.tensor(chorale))

One-hot encode the list

In [23]:
max_note = max(tensor.max().item() for tensor in train_chorales_list + valid_chorales_list + test_chorales_list)
min_note = min(tensor.min().item() for tensor in train_chorales_list + valid_chorales_list + test_chorales_list)
print(f'max note: {max_note}')
print(f'min note: {min_note}')

max note: 81
min note: 0


In [24]:
import torch.nn.functional as F

num_classes = 82

train_chorales_one_hot = []
valid_chorales_one_hot = []
test_chorales_one_hot = []

for chorale in train_chorales_list:
  chorale_one_hot = F.one_hot(chorale, num_classes = num_classes).to(torch.float32)
  train_chorales_one_hot.append(chorale_one_hot)

for chorale in valid_chorales_list:
  chorale_one_hot = F.one_hot(chorale, num_classes = num_classes).to(torch.float32)
  valid_chorales_one_hot.append(chorale_one_hot)

for chorale in test_chorales_list:
  chorale_one_hot = F.one_hot(chorale, num_classes = num_classes).to(torch.float32)
  test_chorales_one_hot.append(chorale_one_hot)

In [66]:
print(f'original shape: {train_chorales_list[0].shape}')
print(f'one-hot shape: {train_chorales_one_hot[0].shape}')

original shape: torch.Size([276, 4])
one-hot shape: torch.Size([276, 4, 82])


In [40]:
train_chorales_one_hot[0].flatten(start_dim=0).shape

torch.Size([90528])

The lists `train_chorales_one_hot` etc are each a list of 4-notes chorales, but we want a list of notes to feed to create the bach dataset using `BachDataset`.

For each chorale in the list:
- flatten it into a single list of notes, each note one-hot encoded so the chorale should have dimension [chorale length, 82]
- create feature and target time windows out of it with `BachDataset`
- concatenate all these windows
- create a `DataLoader`

Time windows of length 128.

In [73]:
from torch.utils.data import ConcatDataset, DataLoader

window_length = 128

datasets = []
for chorale in train_chorales_one_hot:
  flattened_chorale = chorale.flatten(end_dim=1)
  chorale_windows = BachDataset(flattened_chorale, window_length)
  datasets.append(chorale_windows)

train_set = ConcatDataset(datasets)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

datasets = []
for chorale in valid_chorales_one_hot:
  flattened_chorale = chorale.flatten(end_dim=1)
  chorale_windows = BachDataset(flattened_chorale, window_length)
  datasets.append(chorale_windows)

valid_set = ConcatDataset(datasets)
valid_loader = DataLoader(train_set, batch_size=32)

datasets = []
for chorale in test_chorales_one_hot:
  flattened_chorale = chorale.flatten(end_dim=1)
  chorale_windows = BachDataset(flattened_chorale, window_length)
  datasets.append(chorale_windows)

test_set = ConcatDataset(datasets)
test_loader = DataLoader(train_set, batch_size=32)

In [74]:
X, y = train_set[0]
X.shape, y.shape

(torch.Size([128, 82]), torch.Size([128]))

In [77]:
X, y

(tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([62, 55, 47, 67, 62, 55, 47, 67, 62, 55, 47, 67, 62, 55, 47, 67, 63, 60,
         48, 67, 63, 60, 48, 67, 63, 60, 50, 67, 63, 60, 50, 67, 63, 60, 51, 67,
         63, 60, 51, 67, 63, 60, 51, 67, 63, 60, 51, 65, 63, 60, 45, 65, 63, 60,
         45, 65, 63, 60, 45, 65, 63, 60, 45, 67, 62, 55, 47, 67, 62, 55, 47, 67,
         62, 55, 47, 67, 62, 55, 47, 63, 62, 55, 48, 63, 62, 55, 48, 63, 60, 55,
         48, 63, 60, 55, 48, 62, 60, 55, 43, 62, 60, 55, 43, 62, 59, 53, 43, 62,
         59, 53, 43, 60, 60, 51, 48, 60, 60, 51, 48, 60, 60, 51, 48, 60, 60, 51,
         48, 67]))

In [75]:
data, target = next(iter(train_loader))
print("Batch Data Shape:", data.shape)
print("Batch Label Shape:", target.shape)

Batch Data Shape: torch.Size([32, 128, 82])
Batch Label Shape: torch.Size([32, 128])


## Train and evaluation functions

In [29]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.1 MB/s eta 0:00:00


In [82]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)

            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            metric.update(y_pred, y_batch)

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

## Forecasting using a simple RNN model

In [79]:
class SimpleRnnModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    pred = self.output(outputs) # shape: [batch_size, window_size, 82]
    # we want shape [batch_size, 82, window_size]
    return pred.transpose(1,2)

The input_size = number of classes is 82

The output_size = number of classes is 82

In [94]:
torch.manual_seed(42)
bach_model = SimpleRnnModel(input_size =82, hidden_size = 32, output_size=82).to(device)

In [95]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(bach_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=82).to(device)

history = train(bach_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 3.0264, train metric: 0.1516, valid metric: 0.2619
Epoch 2/50, train loss: 2.0645, train metric: 0.3631, valid metric: 0.4802
Epoch 3/50, train loss: 1.5585, train metric: 0.5534, valid metric: 0.6037
Epoch 4/50, train loss: 1.2979, train metric: 0.6383, valid metric: 0.6706
Epoch 5/50, train loss: 1.1302, train metric: 0.6944, valid metric: 0.7168
Epoch 6/50, train loss: 1.0145, train metric: 0.7330, valid metric: 0.7456
Epoch 7/50, train loss: 0.9388, train metric: 0.7542, valid metric: 0.7605
Epoch 8/50, train loss: 0.8958, train metric: 0.7641, valid metric: 0.7673
Epoch 9/50, train loss: 0.8689, train metric: 0.7697, valid metric: 0.7719
Epoch 10/50, train loss: 0.8498, train metric: 0.7736, valid metric: 0.7750
Epoch 11/50, train loss: 0.8356, train metric: 0.7763, valid metric: 0.7774
Epoch 12/50, train loss: 0.8247, train metric: 0.7784, valid metric: 0.7794
Epoch 13/50, train loss: 0.8162, train metric: 0.7800, valid metric: 0.7806
Epoch 14/50, train lo

Look at some predictions

In [96]:
X = valid_set

In [97]:
X[0][0].shape

torch.Size([128, 82])

In [98]:
X[0][0].argmax(dim=-1)

tensor([62, 57, 53, 50, 62, 57, 53, 50, 64, 57, 55, 50, 64, 57, 55, 50, 65, 62,
        57, 50, 65, 62, 57, 50, 65, 62, 59, 50, 65, 62, 59, 50, 67, 64, 60, 48,
        67, 64, 60, 48, 67, 64, 60, 48, 67, 64, 60, 48, 69, 65, 60, 53, 69, 65,
        60, 53, 69, 65, 60, 53, 69, 65, 60, 53, 67, 64, 60, 48, 67, 64, 60, 48,
        67, 64, 59, 48, 67, 64, 59, 48, 72, 64, 57, 48, 72, 64, 57, 48, 72, 64,
        57, 50, 72, 64, 57, 50, 71, 64, 56, 52, 71, 64, 56, 52, 69, 64, 56, 52,
        69, 64, 56, 52, 69, 64, 60, 45, 69, 64, 60, 45, 69, 64, 60, 45, 69, 64,
        60, 45])

In [99]:
bach_model.eval()
with torch.no_grad():
  X = valid_set[0][0].unsqueeze(dim=0)
  Y_pred = bach_model(X.to(device))
print(Y_pred.argmax(dim=1))
print(valid_set[0][1])

tensor([[57, 53, 69, 62, 57, 53, 50, 62, 57, 53, 50, 64, 57, 55, 50, 62, 57, 57,
         50, 65, 62, 57, 50, 65, 62, 57, 50, 65, 62, 59, 50, 65, 62, 59, 48, 67,
         64, 60, 48, 67, 64, 60, 48, 67, 64, 60, 48, 67, 64, 60, 41, 69, 65, 60,
         53, 69, 65, 60, 53, 69, 65, 60, 53, 69, 65, 60, 52, 67, 64, 60, 48, 67,
         64, 60, 48, 67, 64, 60, 48, 67, 64, 57, 48, 72, 64, 57, 48, 72, 64, 57,
         48, 72, 64, 57, 50, 72, 64, 57, 52, 71, 64, 56, 52, 71, 64, 57, 52, 69,
         64, 56, 52, 69, 64, 57, 45, 69, 64, 60, 45, 69, 64, 60, 45, 69, 64, 60,
         45, 69]], device='cuda:0')
tensor([57, 53, 50, 62, 57, 53, 50, 64, 57, 55, 50, 64, 57, 55, 50, 65, 62, 57,
        50, 65, 62, 57, 50, 65, 62, 59, 50, 65, 62, 59, 50, 67, 64, 60, 48, 67,
        64, 60, 48, 67, 64, 60, 48, 67, 64, 60, 48, 69, 65, 60, 53, 69, 65, 60,
        53, 69, 65, 60, 53, 69, 65, 60, 53, 67, 64, 60, 48, 67, 64, 60, 48, 67,
        64, 59, 48, 67, 64, 59, 48, 72, 64, 57, 48, 72, 64, 57, 48, 72, 64, 5